# PadelVision — 02 Ball label bootstrap (Kaggle / Colab)

Phase 2, step 1: find the ball **without a model** and turn it into training labels.

1. Player detections (re-used from notebook 01's run folder if present, otherwise computed here).
2. Classical ball candidates: small moving blobs in the court area, outside player boxes.
3. Trajectory linking: only candidates forming smooth, fast ball flights become **pseudo-labels**.
4. A **review pack** of ~300 frames to label on your laptop with `padelvision label-ball`.

**Setup (Kaggle):** GPU T4 x2 (only needed if player detections must be computed), Internet on,
video attached as a Dataset. Then **Run All** and download `ball_review.zip` from the last cell.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
# court.json: pasted below, else ml/calibrations/<video name>.json in the repo, else court.json next to the video.
COURT_JSON_TEXT = """
"""

court_files = sorted(Path("/kaggle/input" if ON_KAGGLE else "/content/drive/MyDrive/padel").rglob("court.json"))
# Calibrations committed to the repo, named after the video file (e.g. ml/calibrations/Test_video.json).
repo_cal = Path(REPO) / "ml" / "calibrations" / f"{Path(VIDEO_PATH).stem}.json"
if repo_cal.exists():
    court_files.insert(0, repo_cal)
COURT_JSON = f"{WORK}/court.json"
if COURT_JSON_TEXT.strip():
    Path(COURT_JSON).write_text(COURT_JSON_TEXT)
elif court_files:
    Path(COURT_JSON).write_text(court_files[0].read_text())
    print("Using", court_files[0])
else:
    raise FileNotFoundError("No court.json: attach it to the dataset or paste it above")

from padelvision.court import CourtCalibration
cal = CourtCalibration.load(COURT_JSON)
print(f"court: {cal.accuracy_summary()}; lens k1={cal.lens.k1:.3f} k2={cal.lens.k2:.3f}")

In [ ]:
RUN_DIR = f"{WORK}/runs/match1"
MAX_FRAMES = None  # None = whole video

# Player boxes are needed to ignore moving players; re-use notebook 01's detections if they exist.
if not Path(f"{RUN_DIR}/detections.parquet").exists():
    extra = f"--max-frames {MAX_FRAMES}" if MAX_FRAMES else ""
    !cd {REPO} && python -m padelvision.cli analyze "{VIDEO_PATH}" --court {COURT_JSON} --out {RUN_DIR} {extra}
else:
    print("Using existing player detections in", RUN_DIR)

In [ ]:
extra = f"--max-frames {MAX_FRAMES}" if MAX_FRAMES else ""
!cd {REPO} && python -m padelvision.cli ball-bootstrap "{VIDEO_PATH}" --run {RUN_DIR} {extra}

**Pseudo-label check:** crops around 16 pseudo-labels. Each should show the ball in the centre of the ring.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pseudo = pd.read_parquet(f"{RUN_DIR}/ball_pseudo.parquet")
fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS)
print(f"{len(pseudo)} pseudo-labelled frames in {pseudo.track.nunique()} tracks")

sample = pseudo.iloc[np.linspace(0, len(pseudo) - 1, min(16, len(pseudo))).round().astype(int)] if len(pseudo) else pseudo
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
cap = cv2.VideoCapture(VIDEO_PATH)
for ax in axes.ravel():
    ax.axis("off")
for ax, r in zip(axes.ravel(), sample.itertuples()):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(r.frame))
    ok, img = cap.read()
    if not ok:
        continue
    u, v, R = int(r.u), int(r.v), 32
    crop = cv2.copyMakeBorder(img, R, R, R, R, cv2.BORDER_CONSTANT)[v : v + 2 * R, u : u + 2 * R].copy()
    cv2.circle(crop, (R, R), 9, (0, 255, 255), 1)
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), interpolation="nearest")
    ax.set_title(f"{r.frame / fps:.1f}s", fontsize=8)
cap.release()
plt.tight_layout(); plt.show()

per_s = pseudo.groupby((pseudo.frame / fps).astype(int)).size()
plt.figure(figsize=(14, 2.5))
plt.bar(per_s.index, per_s.values, width=1.0)
plt.xlabel("second"); plt.ylabel("pseudo-labelled frames"); plt.title("Where the ball was found over time")
plt.show()

In [ ]:
REVIEW_DIR = f"{WORK}/ball_review"
!cd {REPO} && python -m padelvision.cli ball-review-pack "{VIDEO_PATH}" --run {RUN_DIR} --out {REVIEW_DIR} --n-pseudo 150 --n-random 150

## Download the review pack

Zips the review pack plus the ball files from the run folder into `ball_review.zip`.
Unzip it on your laptop into `C:\PadelVision\data\` and run:

```
.venv/Scripts/padelvision label-ball data/ball_review
```

In [ ]:
import shutil, tempfile
from IPython.display import FileLink

ZIP = f"{WORK}/ball_review.zip"
with tempfile.TemporaryDirectory() as tmp:
    shutil.copytree(REVIEW_DIR, Path(tmp) / "ball_review")
    run_copy = Path(tmp) / "runs" / "match1"
    run_copy.mkdir(parents=True)
    for name in ["ball_candidates.parquet", "ball_pseudo.parquet", "video.json", "court.json"]:
        shutil.copy(Path(RUN_DIR) / name, run_copy / name)
    shutil.make_archive(ZIP[:-4], "zip", tmp)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.1f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()) if Path(ZIP).is_relative_to(Path.cwd()) else ZIP)